In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count

def main():
    spark = (
        SparkSession.builder
        .appName("read_silver_tbcategories")
        .master("local[*]")
        .config("spark.hadoop.fs.defaultFS", "file:///")
        .getOrCreate()
    )
    
    silver_path = "file:/data/silver/tbcategories"
    
    print("Reading Silver table: tbcategories")
    print(f"Path: {silver_path}")
    
    df = spark.read.parquet(silver_path)
    
    print("\nSchema:")
    df.printSchema()
    
    print("\nSample data:")
    df.show(truncate=False)
    
    print("\nTotal records:")
    print(f"Count: {df.count()}")
    
    print("\nRecords per category:")
    (
        df.groupBy("code")
        .agg(count("*").alias("records"))
        .orderBy(col("records").desc())
        .show()
    )
    
    print("\nCreating temp view: tbcategories")
    df.createOrReplaceTempView("tbcategories")
    
    print("\nQuerying all categories:")
    spark.sql("""
        SELECT
            code,
            description
        FROM tbcategories
        ORDER BY code
    """).show(truncate=False)
    
    spark.stop()

if __name__ == "__main__":
    main()

Reading Silver table: tbcategories
Path: file:/data/silver/tbcategories

Schema:
root
 |-- code: integer (nullable = true)
 |-- description: string (nullable = true)


Sample data:
+----+-------------+
|code|description  |
+----+-------------+
|1   |Digital Books|
|2   |Cell Phones  |
|3   |Tablets      |
|4   |Notebooks    |
|5   |Office Supply|
+----+-------------+


Total records:
Count: 5

Records per category:
+----+-------+
|code|records|
+----+-------+
|   1|      1|
|   3|      1|
|   5|      1|
|   4|      1|
|   2|      1|
+----+-------+


Creating temp view: tbcategories

Querying all categories:
+----+-------------+
|code|description  |
+----+-------------+
|1   |Digital Books|
|2   |Cell Phones  |
|3   |Tablets      |
|4   |Notebooks    |
|5   |Office Supply|
+----+-------------+

